In [23]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [24]:
data = pd.read_csv("diabetes.csv")
data.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [25]:
# shape of the data
print("Shape:", data.shape)

# column names
print("\nColumns:")
print(data.columns)

# data types
print("\nData types:")
print(data.dtypes)


Shape: (768, 9)

Columns:
Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

Data types:
Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object


In [26]:
# checking for missing values
data.isnull().sum()


,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [27]:
data.describe()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


No null values, but Glucose, BloodPressure, SkinThickness, Insulin and BMI have 0 values, which is not possible. So treating these 0s as missing values.



In [28]:
# how many zeros in each of these columns
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
(data[zero_cols] == 0).sum()


,0
Glucose,5
BloodPressure,35
SkinThickness,227
Insulin,374
BMI,11


In [29]:
# how many patients have diabetes vs not
data["Outcome"].value_counts()


,count
Outcome,
0,500
1,268


In [30]:
# features and target
X = data.drop("Outcome", axis=1)
y = data["Outcome"]

print("Features:", list(X.columns))
print("Target: Outcome")


Features: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
Target: Outcome


In [31]:
# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (614, 8)
Test size: (154, 8)


In [32]:
# replacing the 0s with the median of that column (median taken from training data only)
X_train = X_train.copy()
X_test = X_test.copy()

for col in zero_cols:
    median_value = X_train[X_train[col] != 0][col].median()
    X_train[col] = X_train[col].replace(0, median_value)
    X_test[col] = X_test[col].replace(0, median_value)

# checking again
print((X_train[zero_cols] == 0).sum())


Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
dtype: int64


In [33]:
# scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Logistic Regression is used because the Outcome is only 0 or 1 (binary classification), and it is a simple model which is easy to understand.



In [34]:
model = LogisticRegression(max_iter=1000)


In [35]:
model.fit(X_train_scaled, y_train)

train_pred = model.predict(X_train_scaled)
print("Training accuracy:", accuracy_score(y_train, train_pred))


Training accuracy: 0.7719869706840391


In [36]:
test_pred = model.predict(X_test_scaled)
print("Test accuracy:", accuracy_score(y_test, test_pred))

print("\nClassification report:")
print(classification_report(y_test, test_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))


Test accuracy: 0.7532467532467533

Classification report:
              precision    recall  f1-score   support

           0       0.80      0.83      0.81        99
           1       0.67      0.62      0.64        55

    accuracy                           0.75       154
   macro avg       0.73      0.72      0.73       154
weighted avg       0.75      0.75      0.75       154

Confusion matrix:
[[82 17]
 [21 34]]


Conclusion: Outcome is the target and the other 8 columns are the features. There were no null values, but some columns had 0 values which is not possible. I split the data into 80% training and 20% testing, replaced the 0 values with the median and scaled the data.

Logistic Regression gave 0.77 training accuracy and 0.75 test accuracy. Both are almost the same, so the model is not overfitting. For the diabetes class, precision is 0.67, recall is 0.62 and F1-score is 0.64.

The model is better at predicting non-diabetic patients. From the confusion matrix, it missed 21 diabetic patients.
